Import Libraries

In [2]:
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage
from dotenv import load_dotenv

load_dotenv()
print("Imports Successful & Ready to go!!!")

Imports Successful & Ready to go!!!


Initialize Claude

In [4]:
llm = ChatAnthropic (
    model = "claude-sonnet-4-6",
    temperature = 0
)
print("Claude is Ready to go as well!!!")

Claude is Ready to go as well!!!


Create the database

In [6]:
import sqlite3
import random
from datetime import datetime, timedelta

# Create database
conn = sqlite3.connect('business.db')
cursor = conn.cursor()

# Create customers table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS customers (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        email TEXT,
        city TEXT,
        state TEXT,
        age INTEGER,
        joined_date TEXT
    )
''')

# Create orders table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS orders (
        id INTEGER PRIMARY KEY,
        customer_id INTEGER,
        product TEXT,
        amount REAL,
        quantity INTEGER,
        order_date TEXT,
        status TEXT,
        FOREIGN KEY (customer_id) REFERENCES customers(id)
    )
''')

# Sample data
customers_data = [
    (1, 'Alice Johnson', 'alice@email.com', 'Austin', 'Texas', 32, '2022-01-15'),
    (2, 'Bob Smith', 'bob@email.com', 'Houston', 'Texas', 45, '2022-03-22'),
    (3, 'Carol White', 'carol@email.com', 'Chicago', 'Illinois', 28, '2022-06-10'),
    (4, 'David Brown', 'david@email.com', 'Phoenix', 'Arizona', 38, '2022-08-05'),
    (5, 'Emma Davis', 'emma@email.com', 'Dallas', 'Texas', 52, '2022-11-18'),
    (6, 'Frank Miller', 'frank@email.com', 'Seattle', 'Washington', 41, '2023-01-09'),
    (7, 'Grace Wilson', 'grace@email.com', 'Miami', 'Florida', 35, '2023-03-14'),
    (8, 'Henry Moore', 'henry@email.com', 'Denver', 'Colorado', 29, '2023-05-20'),
    (9, 'Isabella Taylor', 'isabella@email.com', 'Portland', 'Oregon', 44, '2023-07-11'),
    (10, 'James Anderson', 'james@email.com', 'Austin', 'Texas', 33, '2023-09-25')
]

orders_data = [
    (1, 1, 'Laptop', 1299.99, 1, '2024-01-05', 'Delivered'),
    (2, 1, 'Mouse', 29.99, 2, '2024-01-15', 'Delivered'),
    (3, 2, 'Phone', 899.99, 1, '2024-01-20', 'Delivered'),
    (4, 3, 'Tablet', 499.99, 1, '2024-02-01', 'Delivered'),
    (5, 3, 'Keyboard', 79.99, 1, '2024-02-14', 'Shipped'),
    (6, 4, 'Monitor', 349.99, 2, '2024-02-20', 'Delivered'),
    (7, 5, 'Laptop', 1299.99, 1, '2024-03-01', 'Delivered'),
    (8, 5, 'Headphones', 199.99, 1, '2024-03-10', 'Delivered'),
    (9, 6, 'Phone', 899.99, 2, '2024-03-15', 'Shipped'),
    (10, 7, 'Tablet', 499.99, 1, '2024-04-01', 'Processing'),
    (11, 8, 'Laptop', 1299.99, 1, '2024-04-10', 'Delivered'),
    (12, 9, 'Monitor', 349.99, 1, '2024-04-20', 'Delivered'),
    (13, 10, 'Mouse', 29.99, 3, '2024-05-01', 'Delivered'),
    (14, 2, 'Keyboard', 79.99, 2, '2024-05-10', 'Shipped'),
    (15, 4, 'Headphones', 199.99, 1, '2024-05-20', 'Processing')
]

# Insert data
cursor.executemany('INSERT OR IGNORE INTO customers VALUES (?,?,?,?,?,?,?)', customers_data)
cursor.executemany('INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?,?,?)', orders_data)
conn.commit()
conn.close()

print("Database created successfully!")
print("Tables: customers, orders")
print(f"Customers: {len(customers_data)} records")
print(f"Orders: {len(orders_data)} records")

Database created successfully!
Tables: customers, orders
Customers: 10 records
Orders: 15 records


SQL Tools

In [7]:
import sqlite3

def query_db(sql: str) -> list:
    """Helper function to execute SQL and return results."""
    conn = sqlite3.connect('business.db')
    cursor = conn.cursor()
    cursor.execute(sql)
    results = cursor.fetchall()
    conn.close()
    return results

@tool
def run_sql_query(sql: str) -> str:
    """Executes a SQL SELECT query on the business database and returns results.
    Database has two tables:
    - customers: id, name, email, city, state, age, joined_date
    - orders: id, customer_id, product, amount, quantity, order_date, status
    Write valid SQLite SELECT queries only."""
    try:
        # Safety check - only allow SELECT queries
        if not sql.strip().upper().startswith('SELECT'):
            return "Error: Only SELECT queries are allowed."
        
        results = query_db(sql)
        
        if not results:
            return "No results found for that query."
        
        return f"Query returned {len(results)} rows:\n" + \
               "\n".join([str(row) for row in results])
    except Exception as e:
        return f"SQL Error: {str(e)}"

@tool
def get_schema() -> str:
    """Returns the database schema - table names, columns and data types.
    Always call this first before writing any SQL query."""
    try:
        schema = []
        
        # Get customers schema
        results = query_db("PRAGMA table_info(customers)")
        schema.append("Table: customers")
        schema.append("Columns: " + ", ".join([f"{r[1]} ({r[2]})" for r in results]))
        
        # Get orders schema
        results = query_db("PRAGMA table_info(orders)")
        schema.append("\nTable: orders")
        schema.append("Columns: " + ", ".join([f"{r[1]} ({r[2]})" for r in results]))
        
        # Get row counts
        customers_count = query_db("SELECT COUNT(*) FROM customers")[0][0]
        orders_count = query_db("SELECT COUNT(*) FROM orders")[0][0]
        schema.append(f"\nRecord counts:")
        schema.append(f"  customers: {customers_count} rows")
        schema.append(f"  orders: {orders_count} rows")
        
        return "\n".join(schema)
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_sample_data(table_name: str) -> str:
    """Returns 3 sample rows from a table to understand the data format.
    Valid table names: customers, orders"""
    try:
        if table_name not in ['customers', 'orders']:
            return "Error: Valid tables are 'customers' and 'orders' only."
        
        results = query_db(f"SELECT * FROM {table_name} LIMIT 3")
        return f"Sample data from {table_name}:\n" + \
               "\n".join([str(row) for row in results])
    except Exception as e:
        return f"Error: {str(e)}"

tools = [get_schema, get_sample_data, run_sql_query]
llm_with_tools = llm.bind_tools(tools)
print(f"SQL Tools ready: {[t.name for t in tools]}")

SQL Tools ready: ['get_schema', 'get_sample_data', 'run_sql_query']


ConversationMemory

In [13]:
class SQLConversationMemory:
    def __init__(self):
        self.messages = [
            SystemMessage(content="""You are a SQL agent assistant.
            You have access to tools to get schema, get sample data and run sql queries.
            Always use the appropriate tool to respond for the query asked to you.
            Remember the context of our conversation and refer back to previous results
            when the user ask follow up questions.""")
        ]

    def add_human_message(self, content: str):
        self.messages.append(HumanMessage(content=content))

    def add_ai_message(self, message):
        self.messages.append(message)

    def add_tool_result(self, content: str, tool_call_id: str):
        self.messages.append(
            ToolMessage(content=content, tool_call_id=tool_call_id)
        )
    
    def get_messages(self):
        return self.messages

    def show_history(self):
        print("\n SQL Conversation History:")
        print ("#" * 55)
        for msg in self.messages:
            if isinstance(msg, SystemMessage):
                print(f" System: {msg.content[:55]}...")
            elif isinstance(msg, HumanMessage):
                print(f" You: {msg.content}")
            elif isinstance(msg, AIMessage):
                print(f" Agent: {msg.content}")
            elif isinstance(msg, ToolMessage):
                print(f" Tool Result: {msg.content}")      
        print ("#" * 55)

memory = SQLConversationMemory()
print ("SQL Conversation memory ready!!!")

SQL Conversation memory ready!!!


Agent Loop Function

In [12]:
def run_SQLconversational_agent():
    memory = SQLConversationMemory()
    tool_map = {t.name: t for t in tools}

    print(" SQL Assistant with Necessary Tools Ready!!!")
    print(" You can ask the details about customers & orders queries, order analysis, business insights!!!")
    print(" Type 'history' to see out conversation so far.")
    print(" Type 'quit' to exit.")
    print("#" * 55)

    while True:
        # Get user input
        user_input = input("\n You: ").strip()

        # Handle special commands
        if user_input.lower() == 'quit':
            print("\n Goodbye! Great Querying!!!")
            break

        if user_input.lower() == 'history':
            memory.show_history()
            continue

        if not user_input:
            print("Please enter a question.")
            continue

        # Add user message to memory
        memory.add_human_message(user_input)

        #Get response from Claude
        response = llm_with_tools.invoke(memory.get_messages())
        memory.add_ai_message(response)

        # Process tool calls if any
        while response.tool_calls:
            for tool_call in response.tool_calls:
                print(f"\n Using tool: {tool_call['name']}")
                print(f"   Input: {tool_call['args']}")

                # Execute the tool
                selected_tool = tool_map[tool_call["name"]]
                tool_result = selected_tool.invoke(tool_call["args"])
                print(f"   Result: {tool_result}")

                # Save tool result to memory
                memory.add_tool_result(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            # Get final response after tool execution
            response = llm_with_tools.invoke(memory.get_messages())
            memory.add_ai_message(response)

        print(f"    Agent: {response.content}")
print("SQL Agent loop ready!!!")

SQL Agent loop ready!!!


Run the Agent

In [ ]:
run_SQLconversational_agent()